# Notebook 2 — The SLC26 Superfamily: Subfamilies and Prestin

## Objectives
- Build a phylogenetic tree with **FastTree** and explore it with **ETE4 smartview**
- Extract **all subfamilies** from duplication events in the tree
- Build a **phylogenetic profile** (gene count per species per subfamily) on the NCBI taxonomy
- Compare **alignment statistics** and **branch lengths** across representative subfamilies

## Recap
From Notebook 1 we have a trimmed MAFFT alignment of 297 SLC26 sequences from 30 species.
Today we build the tree, look inside, and characterize the subfamilies.

In [ ]:
import os, subprocess, re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

sns.set_theme(style="whitegrid")

DATA = os.path.join("..", "data")
FIGS = os.path.join("..", "figures")
FASTA = os.path.join(DATA, "selection2_clustalo.fa")
SUB_DIR = os.path.join(DATA, "subfamilies")
os.makedirs(SUB_DIR, exist_ok=True)

def read_fasta(path):
    """Read FASTA → dict {header: sequence}."""
    seqs = {}; header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                header = line[1:].split()[0]; seqs[header] = []
            elif header:
                seqs[header].append(line)
    return {h: "".join(s) for h, s in seqs.items()}

def write_fasta(seqs, path):
    """Write dict {header: sequence} → FASTA file."""
    with open(path, "w") as f:
        for h, s in seqs.items():
            f.write(f">{h}\n{s}\n")

# Gene name mapping (for labeling extracted clades)
seqid2gene = {}
for line in open(os.path.join(DATA, "selection2.clustalo.seqid2gname.tab")):
    parts = line.strip().split("\t")
    if len(parts) >= 2:
        seqid2gene[parts[0]] = parts[1].upper()

# Trimmed alignment from Notebook 1
TRIM = os.path.join(DATA, "slc26.mafft.gt01.fa")
if not os.path.exists(TRIM):
    ALN = os.path.join(DATA, "slc26.mafft.fa")
    if not os.path.exists(ALN):
        print("Running MAFFT...")
        with open(ALN, "w") as f:
            subprocess.run(["mafft", "--auto", "--thread", "-1", FASTA],
                           stdout=f, stderr=subprocess.PIPE, check=True)
    subprocess.run(["trimal", "-in", ALN, "-out", TRIM, "-gt", "0.1", "-fasta"],
                   check=True, capture_output=True)

aln_trim = read_fasta(TRIM)
print(f"Trimmed alignment: {len(aln_trim)} sequences × {len(next(iter(aln_trim.values())))} columns")

---
## 1. Tree inference with FastTree

[FastTree](http://www.microbesonline.org/fasttree/) builds approximately-maximum-likelihood
trees under the **LG** amino acid model. It's fast enough for hundreds of sequences.

In [ ]:
TREE = os.path.join(DATA, "slc26.mafft.gt01.fasttree.nwk")

if not os.path.exists(TREE):
    print("Running FastTree (LG model)...")
    with open(TREE, "w") as f:
        subprocess.run(["FastTree", "-lg", TRIM], stdout=f,
                       stderr=subprocess.PIPE, check=True)
    print("Done.")
else:
    print(f"Cached: {TREE}")

---
## 2. Loading the tree in ETE4

[ETE4](https://etetoolkit.github.io/ete/) extends ETE3 with an interactive
browser-based viewer (**smartview**). Same `PhyloTree` API for taxonomy
annotation and evolutionary event inference, but with a much richer
interactive interface that runs in your browser.

In [ ]:
from ete4 import PhyloTree
from ete4.smartview import Layout, TextFace

t = PhyloTree(open(TREE).read(),
              sp_naming_function=lambda node: node.name.split('.')[0])

t.set_outgroup(t.get_midpoint_outgroup())
t.resolve_polytomy(descendants=True)

print("Annotating with NCBI taxonomy...")
t.annotate_ncbi_taxa()

events = t.get_descendant_evol_events()
n_dup = sum(1 for e in events if e.etype == "D")
n_spec = sum(1 for e in events if e.etype == "S")
print(f"Duplication events: {n_dup}")
print(f"Speciation events:  {n_spec}")
print(f"Leaves: {len(list(t.leaves()))}")

### 2.1 What species do we have?

Each species should have ~10–11 SLC26 genes (A1–A11). Species with fewer
may have gene losses or incomplete genome assemblies.

In [ ]:
species_names = {}
for leaf in t.leaves():
    taxid = leaf.name.split(".")[0]
    sci = leaf.props.get("sci_name", taxid)
    species_names[taxid] = sci

taxid_counts = Counter(leaf.name.split(".")[0] for leaf in t.leaves())

print(f"{'TaxID':<10s} {'Species':<40s} {'Genes':>5s}")
print("-" * 80)
for taxid, count in sorted(taxid_counts.items(), key=lambda x: -x[1]):
    print(f"  {taxid:<10s} {species_names.get(taxid, '?'):<40s} {count:>5d}")

### ✏️ Exercise 1

Which species are **bats**? Which echolocate?
Which are **cetaceans**? Which echolocate?

**Hint:** All Chiroptera except Pteropodidae echolocate. All Odontoceti (toothed whales) echolocate.
```python
from ete4 import NCBITaxa
ncbi = NCBITaxa()
lineage = ncbi.get_lineage(9739)
print([ncbi.get_taxid_translator(lineage)[t] for t in lineage])
```

In [ ]:
# YOUR CODE HERE

---
## 3. Interactive exploration (ETE4 smartview)

ETE4's smartview runs in your browser. We define **layouts** — functions
that return faces (text, shapes) and styles for each node.
Red circles below mark **duplication events** — the boundaries between subfamilies.

In [ ]:
def node_names_style(node):
    if node.is_leaf:
        acc = node.name.split(".")[1] if "." in node.name else node.name
        sci = node.props.get("sci_name", "")
        return [
            TextFace(acc, style={"fill": "black"}, column=0, position="right"),
            TextFace(f"({sci})", style={"fill": "gray"}, column=1, position="right"),
        ]

def node_evo_style(node):
    if node.props.get("evoltype") == "D":
        return {"dot": {"shape": "circle", "radius": 8, "fill": "red"}}

layouts = [
    Layout(name="Names", draw_node=node_names_style),
    Layout(name="Duplications", draw_node=node_evo_style, active=True),
    Layout(name="Style", draw_tree={'node-height-min': 1, 'smart-zoom': False}, active=True),
]

print("Launching ETE4 smartview...")
print("Red circles = duplication events (subfamily boundaries)")
t.explore(layouts=layouts, keep_server=False, port=5006,
          show_leaf_name=False,
          include_props=("name", "sci_name", "evoltype", "dist", "support"))

In the tree:
- **Red circles** mark **duplication events** — the ancient gene duplications
  that created the SLC26 subfamilies (A1–A11) before the mammalian radiation.
- **Within** each subfamily clade: all genes are **orthologs** (same gene,
  different species), connected by speciation events.
- **Between** subfamilies: genes are **paralogs** (different genes that arose
  by duplication in an ancestral genome).

---
## 4. Extracting subfamilies from tree topology

We walk down the tree from the root. At each **duplication** node we
split into children and recurse. When we reach a **speciation** node
(or a leaf), we collect the entire subtree — that's one subfamily.

In [ ]:
def extract_subfamilies(node):
    """Recursively split tree at duplication nodes → list of leaf-name lists."""
    if node.is_leaf:
        return [[node.name]]
    if node.props.get("evoltype") == "D":
        clades = []
        for child in node.children:
            clades.extend(extract_subfamilies(child))
        return clades
    else:
        return [[l.name for l in node.leaves()]]

raw_clades = extract_subfamilies(t)

# Discard very small clades (< 3 leaves) — likely annotation artifacts
subfamilies = [c for c in raw_clades if len(c) >= 3]
singletons = [c for c in raw_clades if len(c) < 3]

print(f"Extracted {len(subfamilies)} clades with ≥3 leaves")
print(f"  ({len(singletons)} small fragments with <3 leaves, discarded)")

### Label subfamilies using gene names

We have a mapping of sequence IDs → gene names. For each clade,
we assign the **most common** gene name (majority vote).

In [ ]:
def label_clade(leaf_names):
    """Assign a subfamily label by majority from known gene names."""
    genes = []
    for name in leaf_names:
        g = seqid2gene.get(name, "")
        m = re.search(r'SLC26A(\d+)', g)
        if m:
            genes.append(f"SLC26A{m.group(1)}")
    if genes:
        return Counter(genes).most_common(1)[0][0]
    return "unknown"

subfamily_dict = {}
for clade in subfamilies:
    label = label_clade(clade)
    if label in subfamily_dict:
        subfamily_dict[label].extend(clade)
    else:
        subfamily_dict[label] = list(clade)

print(f"{'Subfamily':<12s} {'Sequences':>10s}")
print()
for sf in sorted(subfamily_dict, key=lambda x: -len(subfamily_dict[x])):
    print(f"  {sf:<12s} {len(subfamily_dict[sf]):>5d}")

### ✏️ Exercise 2

1. How many subfamilies have exactly one gene per species (1:1 orthologs)?
2. Which subfamilies have **extra copies** in some species (lineage-specific duplications)?
3. Which species have **missing** subfamilies?

Hint: build a species × subfamily count table and check for 0s and >1s.

In [ ]:
# YOUR CODE HERE

---
## 5. Phylogenetic profile on the NCBI species tree

A **phylogenetic profile** shows gene counts per species per subfamily,
displayed alongside the species phylogeny. This reveals gene losses,
expansions, and conserved 1:1 orthology at a glance.

### 5.1 Static heatmap

In [ ]:
all_taxids = sorted(species_names.keys())

profile = pd.DataFrame(0, index=all_taxids,
                       columns=sorted(subfamily_dict.keys(),
                                      key=lambda x: -len(subfamily_dict[x])))

for sf, leaves in subfamily_dict.items():
    for name in leaves:
        taxid = name.split(".")[0]
        if taxid in profile.index and sf in profile.columns:
            profile.loc[taxid, sf] += 1

# Keep a copy with taxid index for ETE4
profile_taxid = profile.copy()

# Add species names for display
profile.index = [f"{tid} ({species_names.get(tid, '?')})" for tid in all_taxids]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(profile, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5,
            cbar_kws={"label": "Gene count"}, ax=ax)
ax.set_title("SLC26 phylogenetic profile")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 5.2 Interactive profile on the NCBI species tree (ETE4)

We build the NCBI species tree for our 30 taxa and attach the gene
counts as an aligned panel — one column per subfamily.

In [ ]:
from ete4 import NCBITaxa

ncbi = NCBITaxa()
tree = ncbi.get_topology([int(t) for t in all_taxids])
ncbi.annotate_tree(tree)
tree.to_ultrametric()

# Attach profile data to leaves
for leaf in tree.leaves():
    taxid = str(leaf.props.get("name", leaf.name))
    if taxid in profile_taxid.index:
        leaf.add_prop("profile", list(profile_taxid.loc[taxid].values.flatten()))

In [ ]:
SF_COLS = list(profile_taxid.columns)
COLORS = {0: "#EEEEEE", 1: "#85B7EB", 2: "#D85A30", 3: "#8B0000"}

def draw_tree_headers(tree):
    return [TextFace(sf, column=i, position="header", rotation=-90,
                     style={"font-size": "10px", "font-weight": "bold"})
            for i, sf in enumerate(SF_COLS)]

def draw_node_profile(node):
    if not node.is_leaf:
        return
    faces = []
    sci = node.props.get("sci_name", node.name)
    faces.append(TextFace(sci, column=0, position="right",
                          style={"fill": "black", "font-style": "italic"}))
    counts = node.props.get("profile", [0] * len(SF_COLS))
    for i, count in enumerate(counts):
        color = COLORS.get(count, "#8B0000")
        faces.append(TextFace(str(count), column=i, position="aligned",
                              style={"fill": color, "font-size": "12px"}))
    return faces

layouts_profile = [
    Layout(name="Headers", draw_tree=draw_tree_headers, active=True),
    Layout(name="Profile", draw_node=draw_node_profile, active=True),
    Layout(name="Style", draw_tree={'node-height-min': 1, 'smart-zoom': False}, active=True),
]

tree.explore(layouts=layouts_profile, keep_server=True, port=5007,
             show_leaf_name=False, name="SLC26 profile")

### ✏️ Exercise 3

From the profile:
1. Which subfamilies are **universally present** (1 copy in all 30 species)?
2. Which species has the **most missing** subfamilies? Why might that be?
3. Are there any **expansions** (>1 copy)? In which subfamily and species?

In [ ]:
# YOUR CODE HERE

---
## 6. Per-subfamily alignment statistics

Different subfamilies evolve at different rates. We pick 5 representative
subfamilies, align each independently with MAFFT, trim, build a FastTree,
and compare basic statistics.

In [ ]:
SELECTED = ["SLC26A1", "SLC26A3", "SLC26A5", "SLC26A6", "SLC26A8"]
all_seqs = read_fasta(FASTA)

In [ ]:
results = {}
for sf in SELECTED:
    if sf not in subfamily_dict:
        continue

    fa = os.path.join(SUB_DIR, f"{sf}.fa")
    aln_f = os.path.join(SUB_DIR, f"{sf}.mafft.fa")
    trim_f = os.path.join(SUB_DIR, f"{sf}.mafft.gt01.fa")
    tree_f = os.path.join(SUB_DIR, f"{sf}.mafft.gt01.fasttree.nwk")

    # Write subfamily FASTA
    sub_seqs = {n: all_seqs[n] for n in subfamily_dict[sf] if n in all_seqs}
    write_fasta(sub_seqs, fa)

    # MAFFT
    if not os.path.exists(aln_f):
        with open(aln_f, "w") as f:
            subprocess.run(["mafft", "--auto", "--thread", "-1", fa],
                           stdout=f, stderr=subprocess.PIPE, check=True)
    # trimAl
    if not os.path.exists(trim_f):
        subprocess.run(["trimal", "-in", aln_f, "-out", trim_f, "-gt", "0.1", "-fasta"],
                       check=True, capture_output=True)
    # FastTree
    if not os.path.exists(tree_f):
        with open(tree_f, "w") as f:
            subprocess.run(["FastTree", "-lg", "-quiet", trim_f], stdout=f, check=True)

    results[sf] = {"tree": tree_f, "trim": trim_f}
    a = read_fasta(trim_f)
    print(f"  {sf}: {len(a)} seqs × {len(next(iter(a.values())))} cols")

### 6.1 Compare alignment statistics

For each subfamily we compute: alignment length, estimated pairwise
identity, mean branch length, and total tree length. Longer total tree
length means faster evolution.

In [ ]:
import random
random.seed(42)

stats = []
for sf, paths in results.items():
    a = read_fasta(paths["trim"])
    n_seqs = len(a)
    aln_len = len(next(iter(a.values())))

    # Pairwise identity (30 random pairs)
    ids_list = list(a.keys())
    identities = []
    for _ in range(min(30, n_seqs * (n_seqs - 1) // 2)):
        i, j = random.sample(ids_list, 2)
        s1, s2 = a[i], a[j]
        matches = sum(1 for x, y in zip(s1, s2) if x == y and x != "-")
        aligned = sum(1 for x, y in zip(s1, s2) if x != "-" and y != "-")
        if aligned > 0:
            identities.append(matches / aligned)

    # Branch lengths from tree
    pt = PhyloTree(open(paths["tree"]).read())
    branch_lengths = [n.dist for n in pt.traverse() if n.dist and n.dist > 0]

    stats.append({
        "Subfamily": sf,
        "Sequences": n_seqs,
        "Aln length": aln_len,
        "Mean identity": f"{np.mean(identities):.1%}" if identities else "?",
        "Mean branch len": f"{np.mean(branch_lengths):.4f}",
        "Total tree len": f"{sum(branch_lengths):.3f}",
    })

df_stats = pd.DataFrame(stats).set_index("Subfamily")
df_stats

### ✏️ Exercise 4

1. Which subfamily has the **highest** pairwise identity? The lowest?
2. Is there a relationship between alignment length and total tree length?
3. What does a longer total tree length tell you about evolutionary rate?
4. Pick two subfamilies and visualize their trees in ETE4. Do the
   topologies match the known species tree?

In [ ]:
# YOUR CODE HERE

### ✏️ Exercise 5

The prestin (SLC26A5) subfamily is the one involved in echolocation.
Extract it and explore the tree — do echolocating species (bats, dolphins)
cluster together?

If not, where is the convergent signal hiding? (Hint: think alignment
positions rather than tree topology — that's what we'll look at tomorrow.)

In [ ]:
# YOUR CODE HERE
# Hint:
# prestin_tree = results["SLC26A5"]["tree"]
# pt = PhyloTree(open(prestin_tree).read(),
#                sp_naming_function=lambda n: n.split('.')[0])
# pt.set_outgroup(pt.get_midpoint_outgroup())
# pt.annotate_ncbi_taxa()
# pt.explore(...)

---
## Summary

- **ETE4 smartview**: interactive tree exploration in the browser
- **Subfamily extraction**: walk down from root, split at duplication nodes → ~11 ortholog groups
- **Phylogenetic profile**: gene count per species per subfamily reveals losses, expansions, and conserved 1:1 orthology
- **Alignment statistics**: subfamilies differ in conservation and evolutionary rate
- **Prestin topology**: echolocators do NOT cluster together in the tree — convergent evolution in prestin operates at the level of individual amino acid sites, not tree topology

**Tomorrow (Day 2):** We detect convergent residues at the alignment level, test significance
with permutation tests, compare prestin to control subfamilies, and ask whether
different tree-building methods are more or less sensitive to convergent signal.